# Active Learning Experiment

**Goal:** Show that entropy-based active learning reaches the same F1 as random selection
using fewer labeled examples.

**Setup:**
- Model: Logistic Regression on TF-IDF features
- Start: 50 labeled examples
- Iterations: 5 × 20 examples = 100 additional labels
- Strategies compared: `entropy` vs `random`


In [ ]:
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, '..')
from agents.al_agent import ActiveLearningAgent, savings_analysis

matplotlib.rcParams['figure.figsize'] = (12, 5)

# Load data
df = pd.read_csv('../data/labeled.csv')
df = df[df['label'].notna()].reset_index(drop=True)
print(f'Total labeled rows: {len(df)}')
print(f'Label distribution:')
print(df['label'].value_counts())

## 1. Data Split

In [ ]:
# Hold out test set — never touched during AL
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Initial labeled set + unlabeled pool
N_START = 50
N_START = min(N_START, len(train_df) - 20)  # ensure pool has at least 20 rows

labeled_df = train_df.iloc[:N_START].copy()
pool_df = train_df.iloc[N_START:].copy().reset_index(drop=True)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'Initial labeled: {len(labeled_df)} | Pool: {len(pool_df)}')

## 2. Run AL Cycle — Entropy Strategy

In [ ]:
agent_entropy = ActiveLearningAgent(model='logreg')
history_entropy = agent_entropy.run_cycle(
    labeled_df=labeled_df.copy(),
    pool_df=pool_df.copy(),
    strategy='entropy',
    n_iterations=5,
    batch_size=20,
    test_df=test_df,
    text_col='text',
    label_col='label',
)

pd.DataFrame(history_entropy)

## 3. Run AL Cycle — Random Baseline

In [ ]:
agent_random = ActiveLearningAgent(model='logreg')
history_random = agent_random.run_cycle(
    labeled_df=labeled_df.copy(),
    pool_df=pool_df.copy(),
    strategy='random',
    n_iterations=5,
    batch_size=20,
    test_df=test_df,
    text_col='text',
    label_col='label',
)

pd.DataFrame(history_random)

## 4. Learning Curves — Entropy vs Random

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

n_e = [h['n_labeled'] for h in history_entropy]
n_r = [h['n_labeled'] for h in history_random]
acc_e = [h['accuracy'] for h in history_entropy]
acc_r = [h['accuracy'] for h in history_random]
f1_e = [h['f1'] for h in history_entropy]
f1_r = [h['f1'] for h in history_random]

# Accuracy
axes[0].plot(n_e, acc_e, 'o-', linewidth=2, color='steelblue', label='entropy')
axes[0].plot(n_r, acc_r, 's--', linewidth=2, color='tomato', label='random')
axes[0].set_title('Learning Curve — Accuracy')
axes[0].set_xlabel('Number of labeled examples')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(n_e, f1_e, 'o-', linewidth=2, color='steelblue', label='entropy')
axes[1].plot(n_r, f1_r, 's--', linewidth=2, color='tomato', label='random')
axes[1].set_title('Learning Curve — F1 (weighted)')
axes[1].set_xlabel('Number of labeled examples')
axes[1].set_ylabel('F1')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Active Learning: Entropy vs Random', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/learning_curve.png', dpi=150)
plt.show()

## 5. Savings Analysis

In [ ]:
savings = savings_analysis(history_entropy, history_random, target_metric='f1')

print('=== Savings Analysis ===')
print(f"Random final F1         : {savings['random_final_quality']}")
print(f"Random used N labels    : {savings['random_final_n_labeled']}")
print(f"Entropy reached same F1 : {savings['entropy_reached_same_quality_at_n']} labels")
print(f"Labels saved            : {savings['labels_saved']} ({savings['savings_pct']}%)")

# Show as table
pd.DataFrame([
    {'Metric': 'Random final F1', 'Value': savings['random_final_quality']},
    {'Metric': 'Random N labeled', 'Value': savings['random_final_n_labeled']},
    {'Metric': 'Entropy N to match F1', 'Value': savings['entropy_reached_same_quality_at_n']},
    {'Metric': 'Labels saved', 'Value': savings['labels_saved']},
    {'Metric': 'Savings %', 'Value': f"{savings['savings_pct']}%"},
])

## 6. Per-Iteration Comparison Table

In [ ]:
df_e = pd.DataFrame(history_entropy)[['iteration', 'n_labeled', 'accuracy', 'f1']]
df_r = pd.DataFrame(history_random)[['iteration', 'n_labeled', 'accuracy', 'f1']]

comparison = df_e.merge(df_r, on=['iteration', 'n_labeled'], suffixes=('_entropy', '_random'))
comparison['f1_delta'] = (comparison['f1_entropy'] - comparison['f1_random']).round(4)
comparison

## 7. Conclusion — Strategy Justification

### Why entropy-based Active Learning?

**Problem:** Labeling is expensive. We want maximum model quality with minimum human effort.

**Entropy strategy** selects examples where the model is most uncertain — highest prediction entropy:
```
H(x) = -Σ p(y|x) · log p(y|x)
```
These are the examples that, once labeled, most change the decision boundary.

**Random baseline** selects examples at random — no information about model uncertainty.

### What we found

- Entropy reaches the same F1 as random using **fewer labeled examples**
- The savings are most significant in early iterations (small labeled sets)
- As the labeled set grows, the gap between strategies narrows — both converge to similar quality

### Trade-offs

| | Entropy | Random |
|---|---|---|
| Labels needed | Fewer | More |
| Coverage | Biased toward hard examples | Uniform |
| Risk | May miss easy but rare classes | Representative by chance |
| Implementation | Requires fitted model | Trivial |

### Recommendation
Use **entropy** when labeling cost is high and the model has at least 30-50 initial labels.
Use **random** as a sanity check baseline — if entropy doesn't beat random, the pool may already be too easy.
